In [ ]:
# Installing Packages for Project - can use % instead of ! if you have any installation issues
!pip install pandas
!pip install numpy
%pip install torch --index-url https://download.pytorch.org/whl/cpu #Downloads CPU based Torch - not full CUDA Libraries aswell
!pip install matplotlib
!pip install scikit-learn
%pip install polars
%pip install pyarrow

In [ ]:
# Importing all packages into the project
import pandas as pd
import numpy as np
import polars as pl
import sklearn
import matplotlib.pyplot as plt
import torch
import pyarrow

In [ ]:
# Parser to read out all Raw Time-Series Data for each user
# Four subdirectories exist under the main raw folder so we go through all 
# Concat each file to the df, we get the Sensor & Device from the filename regex
# Also use filename regex filter to avoid the .DS_Store files

import re 
from pathlib import Path

RAW_DIRECTORY = Path("/home/busing/Documents/Projects/NCP-Wearables/wisdm+smartphone+and+smartwatch+activity+and+biometrics+dataset/wisdm-dataset/raw/") #File path for dataset on device
FILE_NAME = re.compile(r"data_(\d+)_(accel|gyro)_(phone|watch)\.txt$")

print(RAW_DIRECTORY.exists()) # Print confirmation that Notebook can find the filepath
print(len(list(RAW_DIRECTORY.rglob("*.txt")))) # Should print 204 as this is # of .txt files with raw user data

def parse_file(path):
    m = FILE_NAME.search(path.name)
    df = pd.read_csv(path, header = None, 
                    names=["Subject_ID", "Activity_Code", "Timestamp", "x", "y", "z"], 
                     dtype={"Subject_ID": "int", "Activity_Code": "category", "Timestamp": "int64",
                            "x": "float32", "y": "float32", "z": "string"})
    df["z"] = df["z"].str.rstrip(";").astype("float32")
    df["Device"], df["Sensor"] = m.group(3), m.group(2)
    return df

def load_WISDM(raw_dir=RAW_DIRECTORY):
    files = sorted(p for p in Path(raw_dir).rglob("*.txt") if FILE_NAME.search(p.name))
    df = pd.concat([parse_file(p) for p in files], ignore_index = True)
    for c in ("Activity_Code", "Device", "Sensor"): 
        df[c] = df[c].astype("category")
    return df[["Subject_ID", "Activity_Code", "Timestamp", "Device", "Sensor", "x", "y", "z"]]

df = load_WISDM()

df.to_parquet("wisdm_raw.parquet") #Write data to parquet file so we do not need parse data with every kernel restart.


In [ ]:
# Read in data from Parquet File that was created by the parser.
# Much quicker to load in data than parsing with Pandas on each kernel restart.
df = pd.read_parquet("/home/busing/Documents/Projects/NCP-Wearables/wisdm_raw.parquet")
print(df.head()) # Confirm data loaded in after kernel reset

In [ ]:
# Setup "stream" as one device + sensor combination thus there are 4 per participant
# Stream joing Device & Sensor into one label like e.g phone_accel. 
# Since this follows the filename format it allows key to identify each indivdual file from the dataset
KEY = ["Subject_ID", "Stream"]

df["Stream"] = (df["Device"].astype(str) + "_" + df["Sensor"].astype(str)).astype("category")
df["Row_Order"] = df.groupby(KEY, observed = True).cumcount().astype("int32") # Add int counter per file, gives us something to sort by other than timestamp for segmentation

ACTIVITY = {"A": "Walking", "B": "Jogging", "C": "Stairs", "D": "Sitting", "E": "Standing",
            "F": "Typing", "G": "Brushing Teeth", "H": "Eating Soup", "I": "Eating Chips", "J": "Eating Pasta", "K": "Drinking from Cup",
            "L": "Eating Sandwich", "M": "Kicking Soccerball", "O": "Playing Catch with Tennis Ball", "P": "Dribbling Basketball", "Q": "Writing",
            "R": "Clapping", "S": "Folding Clothes"}

EXPECTED_USERS = list(range(1600, 1651))                
EXPECTED_STREAMS = ["phone_accel", "phone_gyro", "watch_accel", "watch_gyro"]
EXPECTED_CODES = list(ACTIVITY)                         

#Check we get the expected count output, should match dataset
print(f"rows: {len(df):,} | participants: {df['Subject_ID'].nunique()} | "
      f"activities: {df['Activity_Code'].nunique()} | streams: {df['Stream'].nunique()}")

In [ ]:
# Count by Participant, Activity & Stream

#Note observed = True ensure we are only getting the combinations that exist in the dataframe not new combinations that are not actually present. 

# Rows per participant per stream
by_user_stream = df.groupby(KEY, observed=True).size().unstack("Stream")
#Creates a table w/ participants as rows and streams as cols. 
#Values are the number of sensor readings per participant
display(by_user_stream)

display(by_user_stream.describe()) # Provides basic summary stats for each stream

# Rows per activity; Col per stream
# .rename(index=ACTIVITY) shows activity w/ name as opposed to with Activity code
by_act_stream = (df.groupby(["Activity_Code", "Stream"], observed=True).size().unstack("Stream").rename(index=ACTIVITY))
display(by_act_stream)

# Rows per participant; Cols per activity (all streams combined)
by_user_act = (df.groupby(["Subject_ID", "Activity_Code"], observed=True).size()
                 .unstack("Activity_Code"))
display(by_user_act)

In [ ]:
# Investigating Missing Files & Missing Activities

present = set(df[KEY].astype({"Stream": str}).drop_duplicates().itertuples(index=False, name=None))
# List Comprehension to do Loop to check for any missing files u = users s = streams
missing_files = [(u, s) for u in EXPECTED_USERS for s in EXPECTED_STREAMS if (u,s) not in present]
print(f"files present: {len(present)} / {len(EXPECTED_USERS) * len(EXPECTED_STREAMS)} ; Missing Files: {missing_files}")

# Expected - Every file has all 18 activities
combo = (df.groupby(KEY + ["Activity_Code"], observed = True).size().rename("rows").reset_index()
        .astype({"Stream": str, "Activity_Code": str}))
# Set of (participant, stream, activity) triples that actually exist in the dataframe
df_has = set(combo[["Subject_ID", "Stream", "Activity_Code"]].itertuples(index=False, name=None))

# The list comprehension builds all 51 x 4 x 18 expected triples and compares to df_has which provides us with the gaps.
missing_activities = pd.DataFrame(
    [(u, s, c, ACTIVITY[c]) for u in EXPECTED_USERS for s in EXPECTED_STREAMS for c in EXPECTED_CODES
     if (u, s, c) not in df_has],
    columns=["Subject_ID", "Stream", "Activity_Code", "Activity"])

print(f"Missing (participant, stream, activity) combinations: {len(missing_activities)}")

display(missing_activities)

In [ ]:
# Investigate Duplicate Records & Timestamps

# keep = first; so that only Second and later copies of a record are flagged as duplicates

# Returns True for any row whose timestape already appeared earlier in the same file
dup_ts = df.duplicated(subset=KEY + ["Timestamp"], keep="first")
# Returns True when a whole row repeats - same activity, timestamp, x, y, z reading
dup_exact = df.duplicated(subset=KEY + ["Activity_Code", "Timestamp", "x", "y", "z"], keep="first") # identical row
# Returns True when timestamp is shared in the same file but the readings (x, y, z) are different
dup_conflict = dup_ts & ~dup_exact 

print(f"duplicate timestamps (extra copies): {dup_ts.sum():,}  ({dup_ts.mean():.2%} of rows)")
print(f"exact duplicate rows: {dup_exact.sum():,}")
print(f"same timestamp, different values: {dup_conflict.sum():,}")

# Lists which files contain duplicates and how many extra rows exist in that file.
dup_summary = (df[dup_ts].groupby(KEY, observed=True).size().rename("extra_rows").reset_index()
                 .query("extra_rows > 0"))
print(f"\nfiles affected: {len(dup_summary)} / {df.groupby(KEY, observed=True).ngroups}")
display(dup_summary)

In [ ]:
# Investigating Timestamps and resets

g = df.groupby(KEY, observed = True) #G represents one file as KEY indicates a Sensor, Device Tuple just as each file does
dt_raw = g["Timestamp"].diff() #Difference between each timestamp and Prev timestamp in file

# True on the first row of each activity block. Compares each row's activity with previous rows activity
new_run = df["Activity_Code"].ne(g["Activity_Code"].shift())  # Find where Activity Changes

# As First Row of New Activity Block occurs during file and not just first line, we need to find the boundary. We also check this point has a valid Timestamp
boundary = new_run & dt_raw.notna()

neg = dt_raw < 0 # Clock Resets

print(f"backwards steps: {neg.sum():,}")
print(f"at an activity boundary: {(neg & boundary).sum():,}")
print(f"inside an activity: {(neg & ~boundary).sum():,}")
print(f"zero steps (repeat timestamp): {(dt_raw == 0).sum():,}")

# Is every activity one contiguous block per file? (more than one block = recording replayed/split)
blocks = new_run.groupby([df["Subject_ID"], df["Stream"], df["Activity_Code"]], observed=True).sum()
print(f"\n(participant, stream, activity) with more than one block: {(blocks > 1).sum()}")
display(blocks[blocks > 1].reset_index(name="blocks").groupby(["Subject_ID", "Stream"], observed=True).size()
        .rename("activities_split").reset_index())


In [ ]:
# Investigating Sampling Rate we actually see in the data & Block Lenghts

d = df.loc[~dup_ts].copy() # Dropped duplicate timestamps so duplicates do not mess with results
g = d.groupby(KEY, observed = True) # as above, just on non-duplicated timestamps
d["new_run"] = d["Activity_Code"].ne(g["Activity_Code"].shift())
d["run_id"] = d.groupby(KEY, observed=True)["new_run"].cumsum() # number of activity block in each file
RUN = KEY + ["run_id"]
d["dt_ms"] = d.groupby(RUN, observed=True)["Timestamp"].diff() / 1e6 # step between samples in m/s within a block (does not measure gap between blocks/activities)

#
runs = (d.groupby(RUN, observed=True).agg(Activity_Code=("Activity_Code", "first"),
                                          n=("Timestamp", "size"), 
                                          t0=("Timestamp", "min"),
                                          t1=("Timestamp", "max")).reset_index())

runs["duration_s"]= (runs["t1"] - runs["t0"]) / 1e9
runs["sampling_rate_hz"] = (runs["n"] - 1) / runs["duration_s"]
runs["rate_class"] = (runs["sampling_rate_hz"] / 5).round() * 5  #nearest 5 Hz

print(f"activity blocks: {len(runs)}  (expected {len(EXPECTED_USERS)*4*18} if complete)")
# Summary Statistics - We should see Central tendacy close to 180s (3 mins) as described in dataset description
display(runs["duration_s"].describe().round(1).to_frame("duration_s (expected ~180)"))
# Displays how many blocks fall into each rate class per stream. Should see most around 20 Hz
display(runs.groupby(["Stream", "rate_class"], observed=True).size().unstack("rate_class", fill_value=0))


In [ ]:
# Activity Blocks where Length is significantly differnet (rouhgly more than 2 S.D from expected) from 3 mins

short_long_blocks = runs[(runs["duration_s"] < 170) | (runs["duration_s"] > 190)]
print(f"Number of blocks 2 standard deviations outside 3 mins: {len(short_long_blocks)}")

#Print a table of all activity blocks where the time is more than 2 S.D from 3 mins
display(short_long_blocks.assign(Activity=lambda t: t["Activity_Code"].astype(str).map(ACTIVITY))
        [KEY + ["Activity", "n", "duration_s", "sampling_rate_hz"]].round(1))


In [ ]:
# Further Investigation of Sampling Rates outside 20 Hz

# Sampling rate per participant per stream - Median across Activity
rate_by_file = runs.groupby(KEY, observed=True)["sampling_rate_hz"].agg(["median", "min", "max"]).round(1)

# Is Hz Median outside Range of 18 Hz - 22 Hz? # Is Hz unstable? Fastest vs Slowest Block Differ by more than 10 Hz
unstable_hz = rate_by_file[(rate_by_file["median"] < 18) | (rate_by_file["median"] > 22) | 
                           (rate_by_file["max"] - rate_by_file["min"] > 10)] 

print(f"files not at a steady 20 Hz : {len(unstable_hz)} / {len(rate_by_file)}")
display(unstable_hz)

In [ ]:
# Investigating Timestamp Gaps & Step Sizes within Blocks where Sample Rates Differ

w = d.dropna(subset=["dt_ms"])

# Loop throuhg cut-off provided cut-off sizes (m/s) for each cut-off provided, count the number of steps that longer than this.
print("steps longer than:")
for t_h in (100, 250, 500, 1000, 5000):
    print(f"{t_h: > 5} ms: {(w['dt_ms']> t_h).sum():,}")

long_gaps = (w.loc[w["dt_ms"] > 500, KEY + ["Activity_Code", "dt_ms"]]
            .assign(Activity=lambda t: t["Activity_Code"].astype(str).map(ACTIVITY)))
display(long_gaps)


# We know there are portions of the data where the sampling rate changes significantly in a stream
# investigation of whether this sampling rate change occurs within an activity block as this will affect number of samples in a window

# Get the 5th Percentile and 95th Percentile for each activity
step_q = w.groupby(RUN, observed=True)["dt_ms"].quantile([0.05, 0.95]).unstack()
# Find Activity Blocks where the 95th percentile is 2x greater than the 5th Percentile
mixed = step_q[step_q[0.95] > 2 * step_q[0.05]].reset_index()

#Print Summary and Table of this data
print(f"\nblocks with mixed step sizes (95% > 2 x 5%): {len(mixed)} / {len(step_q)}")
display(mixed.groupby(KEY, observed=True).size().rename("mixed_blocks").reset_index()
        .sort_values("mixed_blocks", ascending=False).head(15))

In [ ]:
# Sliding Window Size Exploration

WINDOW_SIZE = 10 # Seconds

# For each rate class, how many blocks there are and how many samples a X WINDOW_SIZE window would contain

tbl = (runs.groupby("rate_class")["sampling_rate_hz"].agg(blocks="size", mean_rate="mean").round(1)
       .assign(samples_per_size = lambda t: (t["mean_rate"] * WINDOW_SIZE).round(0)))
display(tbl)

In [ ]:
# Week 2 - Task 1: Segmenetation of Files at Activity Change, Clock Resets and Long Gaps
# Before any within segment sorting. 

# Finding Segmentation Boundaries

GAP_BREAK_S = 1.0 # Step longer than this starts a new segment
RESET_TOLERANCE_S = 1.0 # A backwards step bigger than this is a clock reset

GAP_BREAK_NS = int(GAP_BREAK_S * 1e9) # Timestamps are in Nanoseconds so converting to Nanoseconds
RESET_TOLERANCE_NS = int(RESET_TOLERANCE_S * 1e9)

clone_df = df.copy() # COPY DATAFRAME - We want to leave raw data untouched; can always create a new Parquet file with Segemented & Windowed Data

timestamp = clone_df["Timestamp"].to_numpy(dtype = "int64") 
subject_id = clone_df["Subject_ID"].to_numpy()
stream = clone_df["Stream"].cat.codes.to_numpy() 
activity = clone_df["Activity_Code"].cat.codes.to_numpy()

n = len(clone_df)


#Find the step between each row and the reading before
dt = np.zeros(n, dtype="int64") # Create an array of 0's the length of dataset
dt[1:] = timestamp [1:] - timestamp[:-1] #calculates the time differences between consecutive timestamps in the array

# Each flag is True on the First row of a new segment

new_file = np.ones(n, dtype=bool) # Row 0 is always a new file
new_file[1:] = (subject_id[1:] != subject_id[:-1]) | (stream[1:] != stream[:-1]) # True at any index where the current row has a different subject_id OR a different stream than the previous row.

new_activity = np.zeros(n, dtype = bool) 
new_activity[1:] = activity[1:] != activity [:-1] # If the activity code doesn't match

clock_reset = dt < -RESET_TOLERANCE_NS
long_gap = dt > GAP_BREAK_NS

breaks = new_file | new_activity | clock_reset | long_gap # Breaks are anything that matches one of our flags

# Sum of all break flags 
clone_df["Seg_ID"] = np.cumsum(breaks) - 1 # -1 as 0 index

reason = np.select([new_file, new_activity, clock_reset, long_gap], 
                   ["new_file", "activity_change", "clock_reset", "long_gap"],
                   default = "")
print(f"segments: {clone_df['Seg_ID'].nunique()}")
print(pd.Series(reason[breaks]).value_counts().to_string())
print("(flags can overlap, raw flag count:", f"clock_reset = {clock_reset.sum()}, long_gaps = {long_gap.sum()})")

print(f"breaks from new file: {new_file.sum()}")
print(f"from activity change: {(new_activity & ~new_file).sum()}")
print(f"from clock resets: {clock_reset.sum()}")
print(f"from long gaps: {long_gap.sum()}")


In [ ]:
# Task 1: Sort in Segments, Resolve Duplicates, log conflicts & Drop Replayed Copies

K = ["Seg_ID", "Timestamp"] # Duplicate = same timestamp inside the same segement
VAL = ["x", "y", "z"] 
FILE_TS = ["Subject_ID", "Stream", "Timestamp"] 

EXCLUDE = [] # Exclusion Log 

def log_exclusion(frame, level, reason):
    """Logs excluded data items with their row/segment details and exclusion reason."""
    cols = ["Subject_ID", "Stream", "Seg_ID", "Activity_Code", "n_rows"]
    f = frame.reindex(columns=cols).copy()
    f["level"], f["reason"] = level, reason
    EXCLUDE.append(f)

processed_data = clone_df.sort_values(K, kind = "stable") # stable means ties will keep file ordering

# a) Duplicate timestamps INSIDE a segment
dup_all = processed_data.duplicated(K, keep=False) # Flag every row that is a duplicate including the first occurence, different to how we calculated before. 
dup_rows = processed_data.loc[dup_all, K + VAL] # Drop if duplicate Segment, Timestamp and Values
n_distinct = dup_rows.drop_duplicates().groupby(K).size() # Number distinct value sets per (Seg_ID, Timestamp)
conflict_keys = n_distinct[n_distinct > 1].index # Same timestamp, different value readings

# Log all conflicting duplicate rows to csv file as requested
conflict_log = dup_rows.merge(conflict_keys.to_frame(index=False), on = K)
conflict_log.to_csv("conflict_log.csv", index= False)
print(f"Within-Segment Duplicate Rows: {dup_all.sum()}")
print(f"Within-Segement Duplicate Timestamp and Segement with Conflicting Values: {len(conflict_keys)}")

# Rule for duplicates - exact duplicates we will keep the first; conflicting values keep the first in file order
processed_data = processed_data[~processed_data.duplicated(K, keep = "first")]

# b) Same file + timestamp but different activity code
candidates = processed_data.loc[processed_data.duplicated(FILE_TS, keep = False), FILE_TS + ["Seg_ID", "Activity_Code"]].copy()
candidates["Activity_Code"] = candidates["Activity_Code"].astype(str)
candidates["n_act"] = candidates.groupby(FILE_TS, observed = True)["Activity_Code"].transform("nunique") # Calculates the number of unique activity codes for each distinct timestamp group in DataFrame and assigns result to a new column - n_act

label_conflict = candidates[candidates["n_act"] > 1]
label_conflict.drop(columns="n_act").to_csv("Label_Conflict_Log.csv", index = False)

print(f"Rows with a timestamp shared across different activities: {len(label_conflict)}")
print(label_conflict.groupby(["Subject_ID", "Stream", "Seg_ID", "Activity_Code"], observed = True).size().to_string())

log_exclusion(label_conflict.groupby(["Subject_ID", "Stream", "Seg_ID", "Activity_Code"], observed = True).size().rename("n_rows").reset_index(), "rows", "Label_Conflict_Shared_Timestamp")

processed_data = processed_data.drop(label_conflict.index)

# c) Segment Table & Drop Segments that are exact copies

# Aggregate raw time-series data by segment identifier ('Seg_ID').
# Computes metadata, row counts, boundary timestamps (preserved as int64 nanoseconds),
# and an aggregate signal sum ('x_sum') to serve as a segment fingerprint.

seg = (processed_data.groupby("Seg_ID").agg(Subject_ID = ("Subject_ID", "first"),
                                            Stream = ("Stream", "first"), 
                                            Activity_Code = ("Activity_Code", "first"),
                                            n = ("Timestamp", "size"),
                                            t0 = ("Timestamp", "min"),
                                            t1 = ("Timestamp", "max"),
                                            x_sum = ("x", "sum")))

# Normalise categorical fiedls to strings to ensure we have consistent types for grouping
seg["Stream"] = seg["Stream"].astype(str) 
seg["Activity_Code"] = seg["Activity_Code"].astype(str)

#Define signature coloumns that uniquely identify a segments content and metadata
SIG = ["Subject_ID", "Stream", "Activity_Code", "t0", "t1", "n", "x_sum"]

#Identify duplicated segments; as above we will keep the first and flag any other occurances
dup_seg = seg.duplicated(SIG, keep = "first")
print(f" Number of Duplicated Segments to be dropped : {dup_seg.sum()}")
print(seg[dup_seg].groupby(["Subject_ID", "Stream"]).size().to_string())

log_exclusion(seg[dup_seg].reset_index().assign(n_rows = lambda t: t["n"]), "segment", "duplicated segment") #Logging 

# Removing Duplicates from DataFrame
processed_data = processed_data[~processed_data["Seg_ID"].isin(seg.index[dup_seg])]
seg = seg[~dup_seg].copy() # Prune dups from the aggregated metadata table 'Seg'

#Enum Repeated activity occurences per subject, stream and activity
seg["occ"] = seg.groupby(["Subject_ID", "Stream", "Activity_Code"]).cumcount()
# Deconstruct stream indentifier into individual device and sensore components
seg[["Device", "Sensor"]] = seg["Stream"].str.split("_", expand = True)

print(f"Segment Kept: {len(seg)}")
print(f"(Subject, Stream, Activity) with > 1 occurrence: {(seg['occ'] > 0).sum()}") # Count repeated segment instances (cumcount() is 0-indexed)

In [ ]:
# Week 2 - Task 2: Use Common Time Origin for Accelerometer and Gyro After Verifiying their Clock Relationship
# Propose within device validity rules - at least 160 samples per sensor per 10 second window, observed span >= 8s and no gaps above 0.5 s
# Check these choices on training records and freeze them. 
# Compute statisitics directly on retained samples; no interpolation.

# Check if the accelerometer and gyro on the same clock? 
CLOCK_TOL_S = 0.25       # accel/gyro starts must agree within this to count as "same clock"

PAIR_KEY = ["Subject_ID", "Device", "Activity_Code", "occ"]
acc = seg[seg["Sensor"] == "accel"].reset_index().rename(columns={"Seg_ID": "seg_id_acc"})
gyr = seg[seg["Sensor"] == "gyro" ].reset_index().rename(columns={"Seg_ID": "seg_id_gyr"})

pairs_all = acc.merge(gyr, on=PAIR_KEY, suffixes=("_acc", "_gyro"), how="outer", indicator=True)
print(pairs_all["_merge"].value_counts().to_string())            # 'both' is what we want; others = unpaired segments

# Log Segments with no partner from the other sensor
left_only = pairs_all[pairs_all["_merge"] == "Left_only"]
right_only = pairs_all[pairs_all["_merge"] == "right_only"]

log_exclusion(left_only.assign(Seg_ID=left_only["seg_id_acc"], 
                               Stream = left_only["Stream_acc"],
                               n_rows=left_only["n_acc"]), "segment", "no_gyro_partner_reading")
log_exclusion(right_only.assign(Seg_ID=right_only["seg_id_gyro"],
                                Stream = right_only["Stream_gyro"],
                                n_rows=right_only["n_gyro"]), "Segment", "No_Accel_partner_reading")

pairs = pairs_all[pairs_all["_merge"] == "both"].copy()

# Outer Merge turned readings into float64 (NaNs) Cast back to exact int64 before any maths operations performed

for c in ["t0_acc", "t1_acc", "t0_gyro", "t1_gyro", "seg_id_acc", "seg_id_gyro"]:
    pairs[c] = pairs[c].astype("int64")

# Subtract as int64 FIRST, then convert to seconds this is done as it keeps precision
pairs["start_off_s"] = (pairs["t0_gyr"] - pairs["t0_acc"]) / 1e9
pairs["end_off_s"]   = (pairs["t1_gyr"] - pairs["t1_acc"]) / 1e9

# Find Reading with the same clock, must share the same activity code and occupy the same stretch of time
# Recordings that do not overlap with its partner sensor are considered a wrong pariign, not a clock offset
ov_s = (np.minimum(pairs["t1_acc"], pairs["t1_gyro"]) - np.maximum(pairs["t0_acc"], pairs["t0_gyro"])) / 1e9 # 1e9 to convert to NanoSeconds
shorter_s = np.minimum(pairs["t1_acc"] - pairs["t0_acc"], pairs["t1_gyro"] - pairs["t0_gyro"]) / 1e9

pairs["overlaps"] = ov_s >= 0.5 * shorter_s
pairs["tol_ok"] = (pairs["start_off_s"].abs() <= CLOCK_TOL_S) & (pairs["end_off_s"].abs() <= CLOCK_TOL_S)

print(f"Number Candidate Pairs: {len(pairs)}") 
print(f"Number of Pairs overlapping in time: {pairs['overlaps'].sum()}")
print(f"Within {CLOCK_TOL_S}s tolerance: {pairs['tol_ok'].sum()}")

# Clock Evidence on Overlapping pairs 
good = pairs[pairs["overlaps"]]
print(good.groupby("Device")[["start_off_s", "end_off_s"]].describe(precentiles=[.05,.5,.95]).T.round(4).to_string())

print("Tolerance Failures (overlapping pairs) by device: ")
print(good.groupby("Device")["tol_ok"].agg(pairs="size", failing=lambda s: (~s).sum()).to_string())

bad = pairs[~pairs["overlaps"]]
print(f"Non-Overlapping pairs: {len(bad)}")

display(bad[PAIR_KEY + ["start_off_s", "end_off_s"]])

for sid_col, st_col, n_col in [("seg_id_acc", "Stream_acc", "n_acc"), ("seg_id_gyro", "Stream_gyro", "n_gyro")]:
    log_exclusion(bad.assign(Seg_ID = bad[sid_col], Stream = bad[st_col], n_rows = bad[n_col]), "Segment", "Pair_No_Time_Overlap")

pairs = good.reset_index(drop=True)

In [ ]:
# -Find common origin per accel/gyro pair 
pairs["pair_id"] = np.arange(len(pairs))
pairs["origin_ns"] = np.minimum(pairs["t0_acc"], pairs["t0_gyr"])     # dtype = int64

# Seg_ID to origin_ns lookup, covering both the accel and gyro segment of every pair
seg_origin = pd.concat([pairs.set_index("seg_id_acc")["origin_ns"],
                        pairs.set_index("seg_id_gyr")["origin_ns"]])

assert seg_origin.index.is_unique, "A segment appears in more than one pair"

# Keep only rows that belong to a verified pair. Do this BEFORE mapping so there are no NaNs,
# because NaNs would silently turn the int64 origin into float64 and lose precision.
tw = processed_data[processed_data["Seg_ID"].isin(seg_origin.index)].copy()
origin = tw["Seg_ID"].map(seg_origin).astype("int64")
tw["t_s"] = (tw["Timestamp"] - origin) /1e9 
tw = tw.sort_values(["Seg_ID", "t_s"], kind = "stable").reset_index(drop = True)

print(tw.head())

In [1]:
# Windows & Per Sensor Statistics

WINDOW_S = 10.0
STEP_S = 5.0 

t = tw["t_s"].to_numpy()
seg_arr = tw["Seg_ID"].to_numpy()

# Row Range of every segement inside the sorted array
first = np.flatnonzero(np.r_[True, seg_arr[1:] != seg_arr[:-1]]) #Find the starting indices of consecutive segments where values change
last     = np.r_[first[1:], len(seg_arr)]
bounds   = dict(zip(seg_arr[first], zip(first, last)))


def window_stats(ts_seg, starts): 
    low = np.searchsorted(ts_seg, starts, side = 'left')
    high = np.searchsorted(ts_seg, starts + WINDOW_S, side = "left")
    cnt = high - low
    span = np.zeros(len(starts))
    gap = np.full(len(starts), np.inf)
    #gap/span uses samples inside the window; the span >= 8s rule is what limits empty spaces at the edges
    for i, (a,b) in enumerate (zip(low, high)): 
        if b - a >= 2: 
            s = ts_seg[a:b]
            span[i] = s[-1] - s[0]
            gap[i] = np.diff(s).max()
    return low, high, cnt, span, gap

parts = []

for p in pairs.itertuples():
    a0, a1 = bounds[p.seg_id_acc]
    g0, g1 = bounds[p.seg_id_gyr]
    ta, tg = t[a0:a1], t[g0:g1]
    t_end = max(ta[-1], tg[-1])
    starts = np.arange(0.0, t_end - WINDOW_S + 1e-9, STEP_S)
    if len(starts) == 0:
        continue
    la, ha, na, spa, gpa = window_stats(ta, starts)
    lg, hg, ng, spg, gpg = window_stats(tg, starts)
    parts.append(pd.DataFrame({
            "pair_id" : p.pair_id, "Subject_ID" : p.Subject_ID, "Device": p.Device,
            "Activity_Code": p.Activity_Code, "occ": p.occ, "t_start": starts, "t_start": starts, "tol_ok": p.tol_ok, "acc_seg": p.seg_id_acc, "gyro_seg": p.seg_id_gyro,
            # Row positions in tw. Only valid for this tw, will need to re-run if tw changes
            "acc_a": a0 + la, "acc_b": a0 + ha, "gyr_a": g0 + lg, "gyr_b": g0 + hg, 
            "acc_n": na, "acc_span": spa, "acc_gap": gpa, "gyr_n": ng, "gyr_span": spg, "gyr_gap": gpg}))

windows = pd.concat(parts, ignore_index = True)

# Guards against stable positions i.e the first row of every non-empty window must below to the stated segment
for s, a, n_ in [("acc_seg", "acc_a", "acc_n"), ("gyro_seg", "gyro_a", "gyro_n")]:
    m = windows[n_] > 0
    assert (seg_arr[windows.loc[m, a].to_numpy()] == windows.loc[m, s].to_numpy()).all(), f"stale {a}"

windows["rate_class"] = ((windows["acc_n"]/ WINDOW_S / 5).round() * 5).astype(int) # acc samples/s, nearest 5 Hz

display(windows.head())
print(f"Number of candidate windows: {len(windows)}")

SyntaxError: incomplete input (2219379214.py, line 57)

In [2]:
# Check the Rules on Training Records & Freeze
import json

N_TRAIN, N_TEST, N_EVAL = 31, 10, 10 #Train, test, eval split numbers (by subject ID)


subjects = np.array(sorted(EXPECTED_USERS))
assert len(subjects) == N_TRAIN + N_TEST + N_EVAL

rng = np.random.default_rng(42) # Fixed Seed so the split is reproducible
rng.shuffle(subjects) 

train_split = set(subjects[:N_TRAIN])
eval_split = set(subjects[N_TRAIN: N_TRAIN + N_EVAL])
test_split = set(subjects[N_TRAIN + N_EVAL :]) # Could hardcode in numbers but used constants as best practice and make to code more portable

# Ensure that no Subject_ID appears in more than one of the splits to ensure we don't have data leakage
assert not(train_split & eval_split) and not (train_split & test_split) and not (eval_split & test_split)
assert len(train_split) + len(eval_split) + len(test_split) == 51

# Map each subject to its split label and attach the label
split_map = {**{s: "train" for s in train_split},
             **{s: "eval" for s in eval_split}, 
             **{s: "test" for s in test_split}}
windows["split"] = windows["Subject_ID"].map(split_map) # applying the split map and adding 

print(windows.groupby("split")["Subject_ID"].nunique()) #Should return the split numbers we set in above Constants

# Function for valid windows based on rules proposed 
def valid(w, min_n, min_span, max_gap):
    """ A window is valid only if both sensors of the device pass all three rules"""
    valid_reading = lambda p: ((w[f"{p}_n"] >= min_n) &
                               (w[f"{p}_span"] >= min_span) &
                               (w[f"{p}_gap"] <= max_gap)) 
    return valid_reading("acc") & valid_reading("gyr") # Both sensor must pass

# The proposed rules. 
MIN_N, MIN_SPAN_S, MAX_GAP_S = 160, 8.0, 0.5

windows["valid"] = valid(windows, MIN_N, MIN_SPAN_S, MAX_GAP_S) 
train = windows[windows["split"] == "train"] # ONLY training rows are used to judge the rules

print(f"\nTRAIN windows: {len(train):,} | valid: {train['valid'].mean():.1%}")
print("\npass rate by device:");      
print(train.groupby("Device")["valid"].mean().round(3).to_string())
print("\npass rate by rate class (acc samples/s):")
print(train.groupby("rate_class")["valid"].agg(windows="size", pass_rate="mean").round(3).to_string())
print("\npass rate by activity:")
print(train.groupby("Activity_Code")["valid"].mean().round(3).rename(index=ACTIVITY).to_string())

print("\nwhich rule fails (share of train windows):")
for p in ("acc", "gyr"):
    print(f"  {p}: n<{MIN_N}: {(train[f'{p}_n'] < MIN_N).mean():.3f} | "
          f"span<{MIN_SPAN_S}: {(train[f'{p}_span'] < MIN_SPAN_S).mean():.3f} | "
          f"gap>{MAX_GAP_S}: {(train[f'{p}_gap'] > MAX_GAP_S).mean():.3f}")

print("\nsensitivity (train pass rate):")
for mn, mg in [(140, 0.5), (160, 0.5), (180, 0.5), (160, 0.3), (160, 0.75)]:
    print(f"  min_n={mn}, max_gap={mg}: {valid(train, mn, MIN_SPAN_S, mg).mean():.3f}")

# Freeze: written once the output above has been reviewed. Eval/test are never used to change th
FROZEN_RULES = dict(window_s=WINDOW_S, step_s=STEP_S, min_n=MIN_N,min_span_s=MIN_SPAN_S,max_gap_s=MAX_GAP_S, clock_tol_s=CLOCK_TOL_S,
split_seed=42)

with open("frozen_rules.json", "w") as f:
    json.dump(FROZEN_RU

pd.concat(EXCLUDE, ignore_ "Int64", "n_rows":"Int64"}).to_csv("exclusions.csv", index=False)
print("saved frozen_rules.json and exclusions.csv)


SyntaxError: incomplete input (2888909394.py, line 43)